In [ ]:
import os
import csv
import shutil
import pycdlib
import datetime

In [ ]:
# ---------- Configuration ----------
SOURCE_DIR = r"G:\thecall\done"
DEST_DIR   = r"e:\c_out"
CSV_PATH   = os.path.join(DEST_DIR, "inventory.csv")
ERROR_LOG  = os.path.join(DEST_DIR, "errors.log")

In [ ]:




def log_error(message):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(f"{timestamp} | {message}\n")

def clean_name(name):
    """
    Removes invalid characters and strips trailing spaces/dots 
    which cause [Errno 13] and [WinError 183] on Windows.
    """
    if not name:
        return "unnamed"
    invalid_chars = '<>:"/\\|?*'
    for char in invalid_chars:
        name = name.replace(char, "_")
    # Stripping trailing spaces/dots is critical for Windows folder creation
    return name.strip().strip('.')

def safe_path(path):
    """Prepends the Windows extended length prefix to handle deep paths."""
    if not path.startswith("\\\\?\\"):
        return "\\\\?\\" + os.path.abspath(path)
    return path

def extract_iso(iso_path, out_folder, writer, parent_label):
    iso_name = os.path.basename(iso_path)
    iso = pycdlib.PyCdlib()
    
    try:
        iso.open(iso_path)
    except Exception as e:
        log_error(f"OPEN_FAIL | {iso_path} | {str(e)}")
        # Fallback: Copy the raw ISO if it can't be parsed
        failed_dir = os.path.join(DEST_DIR, "_FAILED_OPEN")
        os.makedirs(safe_path(failed_dir), exist_ok=True)
        shutil.copy2(iso_path, os.path.join(failed_dir, f"{parent_label}_{iso_name}"))
        return

    # Try to find a valid file system within the ISO
    fs_list = [
        ('udf', iso.get_udf_pvd),
        ('joliet', iso.get_joliet_pvd),
        ('iso9660', iso.get_iso9660_pvd)
    ]
    
    active_fs = None
    for fs_name, check_func in fs_list:
        try:
            if check_func():
                active_fs = fs_name
                break
        except:
            continue

    if not active_fs:
        log_error(f"FS_FAIL | {iso_path} | No compatible filesystem (ISO/Joliet/UDF) found.")
        return

    # Walk the ISO filesystem
    search_dir = '/'
    if active_fs == 'joliet':
        walk_iter = iso.walk(joliet=search_dir)
    elif active_fs == 'udf':
        walk_iter = iso.walk(udf=search_dir)
    else:
        walk_iter = iso.walk(iso9660=search_dir)

    for root, dirs, files in walk_iter:
        for name in files:
            # Construct internal ISO path for logging
            iso_internal_path = os.path.join(root, name)
            
            # Clean names for Windows local filesystem
            clean_parts = [clean_name(p) for p in root.split('/') if p]
            local_rel_path = os.path.join(*clean_parts) if clean_parts else ""
            clean_file_name = clean_name(name)
            
            local_dir = os.path.join(out_folder, local_rel_path)
            local_file_path = os.path.join(local_dir, clean_file_name)

            try:
                # Collision Check: If a folder exists where we want a file (or vice versa)
                if os.path.exists(safe_path(local_file_path)):
                    if os.path.isdir(safe_path(local_file_path)):
                        local_file_path += "_file"
                
                os.makedirs(safe_path(local_dir), exist_ok=True)

                # Extraction
                with open(safe_path(local_file_path), "wb") as f:
                    if active_fs == 'joliet':
                        iso.get_file_from_joliet_fp(f, joliet_path=iso_internal_path)
                    elif active_fs == 'udf':
                        iso.get_file_from_udf_fp(f, udf_path=iso_internal_path)
                    else:
                        iso.get_file_from_iso_fp(f, iso_path=iso_internal_path)

                # Write to CSV
                writer.writerow({
                    "iso_file": iso_name,
                    "parent_folder": parent_label,
                    "file_name": clean_file_name,
                    "folder": local_rel_path,
                    "full_path": local_file_path,
                    "path_type": active_fs
                })

            except Exception as e:
                log_error(f"FILE_ERROR | {iso_name} | {iso_internal_path} | {str(e)}")

    iso.close()

def main():
    os.makedirs(safe_path(DEST_DIR), exist_ok=True)
    
    fieldnames = [
        "iso_file", "parent_folder", "file_name", "folder", 
        "full_path", "path_type"
    ]

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, _, files in os.walk(SOURCE_DIR):
            # Capture the numeric label from the folder name
            parent_folder_name = os.path.basename(root)
            
            for file in files:
                if file.lower().endswith(".iso"):
                    iso_path = os.path.join(root, file)
                    iso_stem = os.path.splitext(file)[0]
                    
                    # Output folder: {Number}_{ISO_Name}
                    new_folder_name = f"{parent_folder_name}_{iso_stem}"
                    out_folder = os.path.join(DEST_DIR, clean_name(new_folder_name))
                    
                    print(f"Processing: {parent_folder_name} -> {file}")

                    try:
                        extract_iso(iso_path, out_folder, writer, parent_folder_name)
                    except Exception as e:
                        log_error(f"FATAL | {iso_path} | {str(e)}")

    print(f"Process complete. Check {ERROR_LOG} for details on skipped items.")

if __name__ == "__main__":
    main()